In [1]:
import torch
from unets import UNet

# 1. Setup the Model (matching your LARGO config)
model = UNet(
    image_size=64,
    in_channels=3,
    out_channels=3,
    base_width=64,
    num_classes=102
)

# 2. Create a Dummy Noisy Image, Timestep, and Class Label
# Shape: [Batch, Channels, Height, Width]
x = torch.randn(1, 3, 64, 64) 
t = torch.tensor([500])
y = torch.tensor([42]) # 'Daisy' class

print(f"INPUT SHAPE: {x.shape} (Clear Image + Noise)")

# 3. Manual Trace of the Forward Pass
# This mimics the forward() function in unets.py
hs = []
h = x.float()

# --- ENCODER ---
print("\n--- ENCODER (Downsampling) ---")
# Initial convolution to base width
# (The code in unets.py uses self.input_blocks)
for i, module in enumerate(model.input_blocks):
    h = module(h, model.time_embed(torch.zeros(1, 128))) # Simplified for trace
    hs.append(h)
    print(f"Layer {i} Output: {h.shape}")

# --- BOTTLENECK ---
print("\n--- BOTTLENECK (Deepest Point) ---")
h = model.middle_block(h, model.time_embed(torch.zeros(1, 128)))
print(f"Bottleneck Output: {h.shape}")

# --- DECODER ---
print("\n--- DECODER (Upsampling + Skips) ---")
for i, module in enumerate(model.output_blocks):
    # This is where the magic happens: Concatenating the Encoder 'Memory'
    h = torch.cat([h, hs.pop()], dim=1)
    h = module(h, model.time_embed(torch.zeros(1, 128)))
    print(f"Layer {i} Output: {h.shape}")

final_output = model.out(h)
print(f"\nFINAL OUTPUT SHAPE: {final_output.shape}")

INPUT SHAPE: torch.Size([1, 3, 64, 64]) (Clear Image + Noise)

--- ENCODER (Downsampling) ---


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x128 and 64x256)

In [3]:
metadata = get_metadata("flowers")
metadata

{'image_size': 64,
 'num_classes': 102,
 'train_images': 2040,
 'val_images': 6149,
 'num_channels': 3}

In [35]:
time_embed_dim = metadata.image_size * 4
time_embed = nn.Sequential(
      nn.Linear(metadata.image_size, time_embed_dim),
      nn.SiLU(),
      nn.Linear(time_embed_dim, time_embed_dim),
)

print(f"emb_channels: {time_embed_dim}")

label_emb = nn.Embedding(102, time_embed_dim)
print(label_emb.weight.shape)

channel_mult = (1, 2, 3, 4)

ch = input_ch = int(channel_mult[0] * 64)
print(f"ch {ch}")
conv = nn.Conv3d(2, input_ch, ch, 3, padding=1)
input_blocks = nn.ModuleList(
      [nn.Sequential(conv)]
)


emb_channels: 256
torch.Size([102, 256])
ch 64


In [44]:
input = th.randn(20, 6, 10, 10)
print(input.shape)
# Separate 6 channels into 3 groups
m = nn.GroupNorm(3, 6)
# Activating the module
output = m(input)
print(output.weighs)
print(th.equal(input, output))

torch.Size([20, 6, 10, 10])


AttributeError: 'Tensor' object has no attribute 'weighs'

In [ ]:

for level, mult in enumerate(channel_mult):
      for _ in range(3):
            layers = [
                    ResBlock(
                        64,
                        time_embed_dim,
                        0.1,
                        out_channels=int(mult * 64),
                        dims=2,
                        use_checkpoint=False,
                        use_scale_shift_norm=False,
                    )
                ]

In [5]:
attention_ds = []
attention_resolutions = "32,16,8"
for res in attention_resolutions.split(","):
      attention_ds.append(metadata.image_size // int(res))
attention_ds

[2, 4, 8]

In [9]:
x = th.tensor([[[[1., 2.],
                    [3., 4.]]]])

print("--- Original Image (2x2 grid) ---")
print(x)
print(f"Shape: {x.shape}")

flat_x = x.reshape(1, 1, -1)
print("\n--- Flattened (Line) ---")
print(flat_x)
print(f"Shape: {flat_x.shape}")

--- Original Image (2x2 grid) ---
tensor([[[[1., 2.],
          [3., 4.]]]])
Shape: torch.Size([1, 1, 2, 2])

--- Flattened (Line) ---
tensor([[[1., 2., 3., 4.]]])
Shape: torch.Size([1, 1, 4])


In [10]:
mean_x = flat_x.mean(dim=-1, keepdim=True)
mean_x


tensor([[[2.5000]]])